# 02 · RAG Strategy Comparison

Comparação das três estratégias de retrieval implementadas em `src/llm_eval/rag/`:
**Naive**, **HyDE** e **Reranking**.

**Métricas RAGAS:** `faithfulness`, `answer_relevancy`, `context_recall`

**Input:** `scripts/results/rag_summary.json` — gerado por `uv run evaluate-rag --all-strategies`

**Output:** gráfico de barras, radar chart, trade-off qualidade × latência

> **Pré-requisito:**
> ```bash
> uv run evaluate-rag --all-strategies
> ```

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 130

# Localiza results/ de forma robusta — funciona rodando de notebooks/ ou da raiz
NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

RESULTS_DIR = ROOT / 'scripts' / 'results'
SUMMARY_PATH = RESULTS_DIR / 'rag_summary.json'

print('Raiz do projeto :', ROOT)
print('Summary path    :', SUMMARY_PATH)
print('Summary existe  :', SUMMARY_PATH.exists())
print('CSVs RAG        :', sorted(RESULTS_DIR.glob('rag_*.csv')))

Raiz do projeto : C:\Users\otnie\OneDrive\Documentos\GitHub\LLM-Eval-Suite
Summary path    : C:\Users\otnie\OneDrive\Documentos\GitHub\LLM-Eval-Suite\scripts\results\rag_summary.json
Summary existe  : False
CSVs RAG        : []


## 1 · Carregamento dos resultados RAGAS

In [2]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f'Arquivo não encontrado: {SUMMARY_PATH}\n'
        'Rode: uv run evaluate-rag --all-strategies'
    )

with open(SUMMARY_PATH) as f:
    summaries = json.load(f)

df = pd.DataFrame(summaries)

# Normaliza nomes de colunas
rename_map = {
    'latency_mean': 'latency_ms_mean',
    'avg_latency_ms': 'latency_ms_mean',
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

print('Colunas:', df.columns.tolist())
print('Estratégias:', df['strategy'].unique().tolist())

RAGAS_METRICS = [c for c in ['faithfulness', 'answer_relevancy', 'context_recall'] if c in df.columns]
print('Métricas RAGAS disponíveis:', RAGAS_METRICS)

df[['strategy'] + RAGAS_METRICS + (['latency_ms_mean'] if 'latency_ms_mean' in df.columns else [])]

FileNotFoundError: Arquivo não encontrado: C:\Users\otnie\OneDrive\Documentos\GitHub\LLM-Eval-Suite\scripts\results\rag_summary.json
Rode: uv run evaluate-rag --all-strategies

## 2 · Barras comparativas por métrica RAGAS

In [ ]:
melted = df.melt(
    id_vars='strategy',
    value_vars=RAGAS_METRICS,
    var_name='metric',
    value_name='score'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=melted,
    x='metric', y='score', hue='strategy',
    ax=ax, width=0.6
)
ax.set_ylim(0, 1.1)
ax.set_title('Métricas RAGAS por Estratégia de Retrieval', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score (0–1)')
ax.legend(title='Estratégia')
plt.tight_layout()
plt.savefig('rag_metrics_bar.png', bbox_inches='tight')
plt.show()

## 3 · Radar chart — perfil de cada estratégia

In [ ]:
labels = [m.replace('_', '\n') for m in RAGAS_METRICS]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

COLOR_MAP = {'naive': '#4C72B0', 'hyde': '#DD8452', 'reranking': '#55A868'}

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for _, row in df.iterrows():
    values = [row[m] for m in RAGAS_METRICS]
    values += values[:1]
    strategy = row['strategy']
    color = COLOR_MAP.get(strategy, 'gray')
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=strategy)
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('Radar — Perfil das Estratégias RAG', fontweight='bold', pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), title='Estratégia')
plt.tight_layout()
plt.savefig('rag_radar.png', bbox_inches='tight')
plt.show()

## 4 · Trade-off: qualidade × latência

In [ ]:
df['composite_score'] = df[RAGAS_METRICS].mean(axis=1)

if 'latency_ms_mean' not in df.columns:
    print('Coluna latency_ms_mean não encontrada — pulando gráfico de trade-off.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    for _, row in df.iterrows():
        color = COLOR_MAP.get(row['strategy'], 'gray')
        ax.scatter(
            row['latency_ms_mean'], row['composite_score'],
            s=220, color=color, zorder=3, edgecolors='white', linewidths=1.5
        )
        ax.annotate(
            row['strategy'],
            xy=(row['latency_ms_mean'], row['composite_score']),
            xytext=(8, 4), textcoords='offset points', fontsize=11
        )

    ax.set_xlabel('Latência Média (ms)', fontsize=12)
    ax.set_ylabel('Score Composto RAGAS', fontsize=12)
    ax.set_title('Trade-off: Qualidade × Latência por Estratégia', fontweight='bold')
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig('rag_tradeoff.png', bbox_inches='tight')
    plt.show()

## 5 · Distribuição por questão — context_recall

In [ ]:
dfs_per_q = []
for path in sorted(RESULTS_DIR.glob('rag_*.csv')):
    strategy_name = path.stem.replace('rag_', '')
    tmp = pd.read_csv(path)
    tmp['strategy'] = strategy_name
    dfs_per_q.append(tmp)

if dfs_per_q:
    all_q = pd.concat(dfs_per_q, ignore_index=True)
    recall_col = 'context_recall' if 'context_recall' in all_q.columns else None

    if recall_col:
        fig, ax = plt.subplots(figsize=(9, 4))
        sns.violinplot(
            data=all_q, x='strategy', y=recall_col,
            ax=ax, palette='Set2', inner='box', cut=0
        )
        ax.set_title('Distribuição de Context Recall por Questão', fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Context Recall (0–1)')
        ax.set_ylim(-0.05, 1.1)
        plt.tight_layout()
        plt.savefig('rag_recall_dist.png', bbox_inches='tight')
        plt.show()
    else:
        print('Coluna context_recall não encontrada nos CSVs.')
else:
    print('CSVs individuais não encontrados em', RESULTS_DIR)
    print('Rode: uv run evaluate-rag --all-strategies')

## 6 · Ranking final

In [ ]:
# Garante que composite_score existe
if 'composite_score' not in df.columns:
    df['composite_score'] = df[RAGAS_METRICS].mean(axis=1)

cols_present = [c for c in
    ['strategy', 'faithfulness', 'answer_relevancy', 'context_recall',
     'composite_score', 'latency_ms_mean']
    if c in df.columns
]

ranking = (
    df[cols_present]
    .sort_values('composite_score', ascending=False)
    .rename(columns={
        'strategy': 'Estratégia',
        'faithfulness': 'Faithfulness',
        'answer_relevancy': 'Answer Relevancy',
        'context_recall': 'Context Recall',
        'composite_score': 'Score Composto',
        'latency_ms_mean': 'Latência (ms)',
    })
    .set_index('Estratégia')
    .round(4)
)

quality_cols = [c for c in ['Faithfulness', 'Answer Relevancy', 'Context Recall', 'Score Composto'] if c in ranking.columns]
latency_cols = [c for c in ['Latência (ms)'] if c in ranking.columns]

styler = ranking.style.format(precision=4).set_caption('Ranking Final — Estratégias de Retrieval RAG')  # type: ignore[arg-type]
if quality_cols:
    styler = styler.background_gradient(cmap='Greens', subset=quality_cols)  # type: ignore[arg-type]
if latency_cols:
    styler = styler.background_gradient(cmap='Reds_r', subset=latency_cols)  # type: ignore[arg-type]
styler

## 7 · Conclusões

Preencha após executar `uv run evaluate-rag --all-strategies`:

- **Melhor faithfulness:** `...` — respostas mais ancoradas no contexto
- **Melhor context_recall:** `...` — recupera mais informação relevante
- **Melhor trade-off qualidade/latência:** `...`
- **Insight HyDE:** melhora recall em queries abstratas, mas pode introduzir viés na geração hipotética
- **Insight Reranking:** melhora precision, mas adiciona ~N × latência do LLM judge
- **Recomendação para produção:** `...`